In [13]:
import json 
import pandas as pd 
import gurobipy as gp
from src.build_recipe_matrix import load_recipe_data


In [14]:

recipe_df, dishes, ingredients, recipes = load_recipe_data(recipe_data="data/recipes.json", save_csv=True)
dishes = [dish for dish, _ in recipes]

import json

with open("./data/hp_data.json", "r") as file:
    hp_data = json.load(file)

In [15]:
MAX_HP = 50

hp = {
    dish: data["percent"] * MAX_HP + data["flat"]
    for dish, data in hp_data.items()
}



In [16]:

with open("./data/inventory.json", "r") as file:
    Inventory = json.load(file)

In [17]:
import gurobipy as gp
model = gp.Model("Food production plan")
# model.setParam("OutputFlag", 0)


hp_value = {
    dish: data["percent"] * 100 + data["flat"]
    for dish, data in hp_data.items()
}

x = model.addVars(
    dishes,
    vtype=gp.GRB.INTEGER,
    #vtype=gp.GRB.CONTINUOUS,
    lb=0,
    name="x"
)

In [18]:
model.setObjective(
    gp.quicksum(
        hp_value[dish] * x[dish]
        for dish in dishes
    ), 
    gp.GRB.MAXIMIZE
)



In [19]:
constraints = {}

for ingredient in recipe_df.index:

    constraints[ingredient] = model.addConstr(
        gp.quicksum(
            recipe_df.loc[ingredient, dish] * x[dish]
            for dish in dishes
        )
        <= Inventory[ingredient],

        name=f"{ingredient}_inventory"
    )

In [20]:
model.write("food_allocation.lp")

In [21]:
model.optimize()
if model.status == gp.GRB.OPTIMAL:

    print("Maximum HP:", model.ObjVal)

    for dish in dishes:
        if x[dish].X > 0:
            print(
                dish,
                round(x[dish].X)
            )
    baseline_hp = model.ObjVal

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 23.2.0 23C64)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 38 rows, 34 columns and 99 nonzeros (Max)
Model fingerprint: 0x3b10d46e
Model has 34 linear objective coefficients
Variable types: 0 continuous, 34 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [6e+02, 4e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+00, 8e+02]

Found heuristic solution: objective 250684.00000
Presolve removed 32 rows and 18 columns
Presolve time: 0.00s
Presolved: 6 rows, 16 columns, 24 nonzeros
Found heuristic solution: objective 275176.00000
Variable types: 0 continuous, 16 integer (1 binary)

Root relaxation: objective 3.694810e+05, 7 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    Best

# Sensitivity analysis

In [23]:
sensitivity_model = gp.Model("Food Resource Allocation - Sensitivity")

sensitivity_x = sensitivity_model.addVars(
    dishes,
    vtype=gp.GRB.CONTINUOUS,
    lb=0,
    name="x"
)

sensitivity_model.setObjective(
    gp.quicksum(
        hp_value[dish] * sensitivity_x[dish]
        for dish in dishes
    ),
    gp.GRB.MAXIMIZE
)

sensitivity_constraints = {}

for ingredient in recipe_df.index:
    sensitivity_constraints[ingredient] = sensitivity_model.addConstr(
        gp.quicksum(
            recipe_df.loc[ingredient, dish] * sensitivity_x[dish]
            for dish in dishes
        )
        <= Inventory[ingredient],
        name=f"{ingredient}_inventory"
    )

sensitivity_model.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 23.2.0 23C64)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 38 rows, 34 columns and 99 nonzeros (Max)
Model fingerprint: 0xadbd81f0
Model has 34 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [6e+02, 4e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+00, 8e+02]

Presolve removed 32 rows and 19 columns
Presolve time: 0.00s
Presolved: 6 rows, 15 columns, 24 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.3134400e+05   2.020821e+01   0.000000e+00      0s
       3    3.7005433e+05   0.000000e+00   0.000000e+00      0s

Solved in 3 iterations and 0.00 seconds (0.00 work units)
Optimal objective  3.700543333e+05


In [24]:
if sensitivity_model.status == gp.GRB.OPTIMAL:

    print(f"\nContinuous LP Maximum HP: {sensitivity_model.ObjVal:,.2f}\n")
    print(
        f"{'Ingredient':<20} {'Shadow Price':>13} {'Slack':>8} "
        f"{'Inventory':>10} {'RHS Low':>10} {'RHS Up':>10}"
    )
    print("-" * 75)

    for ingredient, constraint in sorted(
        sensitivity_constraints.items(),
        key=lambda x: x[1].Pi,
        reverse=True
    ):
        print(
            f"{ingredient:<20} {constraint.Pi:>13.2f} {constraint.Slack:>8.1f} "
            f"{constraint.RHS:>10.1f} {constraint.SARHSLow:>10.1f} {constraint.SARHSUp:>10.1f}"
        )


Continuous LP Maximum HP: 370,054.33

Ingredient            Shadow Price    Slack  Inventory    RHS Low     RHS Up
---------------------------------------------------------------------------
Sugar                      3748.00      0.0        5.0        0.0        8.3
Flour                      1268.00      0.0        7.0        0.0      467.0
Radish                     1160.25      0.0      105.0       87.7      111.7
Salt                       1012.00      0.0       54.0       33.0      100.0
Fowl                        808.00      0.0      100.0       54.0      121.0
Fish                        700.25      0.0       26.0        8.7       32.7
Rice                        630.00      0.0       10.0        0.0       36.5
Matsutake                   460.00      0.0        5.0        0.0      109.7
Pepper                      460.00      0.0       61.0        1.7      119.7
Jam                         315.00      0.0       10.0        5.0       23.0
Potato                      315.00    